# Experiment 1: Time and Frequency Domain, and the Pendulum
**Course:** CSAI4014 Algorithms for Intelligent Systems and Robotics  
**Student:** Aadi Chopra | SAP ID: 500120163 | Roll No: R2142231902 | Batch: B2 AIML  
**Repository:** https://github.com/aadichopra/aisr-lab-500120163  

## 1. Aim
To represent a sampled signal in the time domain and in the frequency domain using the discrete Fourier transform, and to simulate the motion of a simple pendulum by numerical integration of its equation of motion.


In [ ]:
import os
import numpy as np
import scipy.integrate as integrate
import matplotlib.pyplot as plt

os.makedirs('figs', exist_ok=True)

# 1. Signal Parameters
fs = 500.0   # Sampling frequency (Hz)
T = 2.0      # Record length (seconds)
dt = 1.0 / fs
t = np.arange(0, T, dt)
N = len(t)
df = fs / N
print(f"Sampling Parameters: fs = {fs} Hz, T = {T} s, N = {N}, dt = {dt*1000:.2f} ms, df = {df:.2f} Hz, Nyquist = {fs/2:.1f} Hz")

# Synthesize multi-tone signal with Gaussian noise
rng = np.random.default_rng(0)
x = (1.00 * np.sin(2 * np.pi * 5 * t) + 
     0.50 * np.sin(2 * np.pi * 20 * t) + 
     0.25 * np.sin(2 * np.pi * 50 * t) + 
     0.20 * rng.standard_normal(N))

# Plot Figure 1.1: Time Domain Signal
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=150)
ax.plot(t[:250], x[:250], color='#1f77b4', linewidth=1.2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Signal Amplitude')
ax.set_title('Synthesized Multi-Tone Signal in Time Domain (First 0.5 s)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('figs/fig1_1_signal.png')
plt.show()


In [ ]:
# 2. Discrete Fourier Transform (One-Sided Spectrum)
X = np.fft.rfft(x)
f = np.fft.rfftfreq(N, d=dt)
amp = np.abs(X) * 2.0 / N
amp[0] /= 2.0

for target_f in [5.0, 20.0, 50.0]:
    idx = int(round(target_f / df))
    print(f"Set Frequency: {target_f:4.1f} Hz -> Measured: {f[idx]:4.1f} Hz, Measured Amplitude: {amp[idx]:.3f}")

# Plot Figure 1.2: Frequency Spectrum
fig, ax = plt.subplots(figsize=(6.5, 3.2), dpi=150)
ax.plot(f, amp, color='#2ca02c', linewidth=1.2)
ax.set_xlim(0, 100)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Amplitude')
ax.set_title('One-Sided Amplitude Spectrum (Peaks at 5, 20, and 50 Hz)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.6)
for target_f, set_amp in [(5.0, 1.00), (20.0, 0.50), (50.0, 0.25)]:
    idx = int(round(target_f / df))
    ax.annotate(f"{f[idx]:.0f} Hz ({amp[idx]:.2f})", xy=(f[idx], amp[idx]),
                xytext=(f[idx]+2, amp[idx]+0.08),
                arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=4))
plt.tight_layout()
plt.savefig('figs/fig1_2_spectrum.png')
plt.show()


In [ ]:
# 3. Aliasing Demonstration (5 Hz sinusoid sampled at 8 Hz)
fs_low = 8.0
t_low = np.arange(0, T, 1.0 / fs_low)
N_low = len(t_low)
x_low = np.sin(2 * np.pi * 5.0 * t_low)
X_low = np.fft.rfft(x_low)
f_low = np.fft.rfftfreq(N_low, d=1.0 / fs_low)
amp_low = np.abs(X_low) * 2.0 / N_low

apparent_idx = np.argmax(amp_low[1:]) + 1
print(f"True Frequency: 5.0 Hz sampled at fs = {fs_low} Hz (< 2 * 5.0 Hz)")
print(f"Apparent Aliased Peak: {f_low[apparent_idx]:.1f} Hz (|8 - 5| = 3.0 Hz)")

# Plot Figure 1.3: Aliasing Effect
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.5, 4.2), dpi=150)
t_fine = np.linspace(0, 1.0, 500)
ax1.plot(t_fine, np.sin(2 * np.pi * 5.0 * t_fine), 'gray', linestyle=':', label='True 5 Hz signal')
ax1.plot(t_low[:9], x_low[:9], 'ro-', label='Sampled at 8 Hz (dt=0.125s)')
ax1.plot(t_fine, np.sin(2 * np.pi * 3.0 * t_fine), 'b--', label='Apparent 3 Hz alias')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Amplitude')
ax1.legend(loc='upper right', fontsize=8)
ax1.set_title('Aliasing in Time Domain: 5 Hz sampled at 8 Hz appears as 3 Hz', fontsize=10)
ax1.grid(True, linestyle='--', alpha=0.6)

ax2.stem(f_low, amp_low, linefmt='r-', markerfmt='ro', basefmt='k-')
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('Amplitude')
ax2.set_title('Spectrum of Undersampled Signal Showing Alias at 3.0 Hz', fontsize=10)
ax2.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('figs/fig1_3_aliasing.png')
plt.show()


In [ ]:
# 4. Numerical Integration of Simple Pendulum
g, L = 9.81, 1.0
T0 = 2 * np.pi * np.sqrt(L / g)
print(f"Small-angle linear period T0: {T0:.4f} s")

def pendulum_ode(t, state, b):
    theta, omega = state
    return [omega, -b * omega - (g / L) * np.sin(theta)]

t_eval = np.linspace(0, 10, 1000)
sol_undamped = integrate.solve_ivp(pendulum_ode, (0, 10), [np.radians(20), 0.0], args=(0.0,), t_eval=t_eval)
sol_under = integrate.solve_ivp(pendulum_ode, (0, 10), [np.radians(20), 0.0], args=(0.5,), t_eval=t_eval)
sol_over = integrate.solve_ivp(pendulum_ode, (0, 10), [np.radians(20), 0.0], args=(4.0,), t_eval=t_eval)

# Plot Figure 1.4: Damping Regimes
fig, ax = plt.subplots(figsize=(6.5, 3.4), dpi=150)
ax.plot(t_eval, np.degrees(sol_undamped.y[0]), label='Undamped (b = 0.0)', color='#1f77b4', linewidth=1.5)
ax.plot(t_eval, np.degrees(sol_under.y[0]), label='Underdamped (b = 0.5)', color='#2ca02c', linewidth=1.5)
ax.plot(t_eval, np.degrees(sol_over.y[0]), label='Overdamped (b = 4.0)', color='#d62728', linewidth=1.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle $\theta$ (degrees)')
ax.set_title('Simple Pendulum Response under Different Damping Regimes ($\theta_0 = 20^\circ$)', fontsize=11)
ax.legend(loc='upper right')
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('figs/fig1_4_pendulum_damping.png')
plt.show()


In [ ]:
# 5. Nonlinear Period vs. Initial Amplitude & Phase Portrait
amplitudes_deg = [10, 20, 30, 45, 60, 90]
measured_periods = []

for amp_deg in amplitudes_deg:
    theta0 = np.radians(amp_deg)
    sol = integrate.solve_ivp(pendulum_ode, (0, 20), [theta0, 0.0], args=(0.0,), t_eval=np.linspace(0, 20, 5000), rtol=1e-8, atol=1e-8)
    # Zero crossings with positive velocity
    signs = np.sign(sol.y[0])
    crossings = np.where(np.diff(signs) > 0)[0]
    periods = np.diff(sol.t[crossings])
    measured_periods.append(np.mean(periods))
    print(f"Amplitude: {amp_deg:2d} deg -> Measured Period: {measured_periods[-1]:.4f} s (Ratio T/T0: {measured_periods[-1]/T0:.4f})")

# Plot Figure 1.5: Phase Portrait and Period vs Amplitude
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 3.3), dpi=150)

# Left: Phase Portrait
for amp_deg in [20, 60, 90]:
    sol = integrate.solve_ivp(pendulum_ode, (0, 10), [np.radians(amp_deg), 0.0], args=(0.0,), t_eval=np.linspace(0, 10, 1000))
    ax1.plot(sol.y[0], sol.y[1], label=f'$\theta_0 = {amp_deg}^\circ$')
ax1.set_xlabel('$\theta$ (rad)'); ax1.set_ylabel('$\dot{\theta}$ (rad/s)')
ax1.set_title('Phase Portrait (Undamped)', fontsize=10)
ax1.legend(loc='upper right', fontsize=8)
ax1.grid(True, linestyle='--', alpha=0.6)

# Right: Period Lengthening
ax2.plot(amplitudes_deg, measured_periods, 'bo-', label='Empirical RK45')
ax2.axhline(T0, color='red', linestyle='--', label=f'Linear $T_0 = {T0:.4f}$ s')
ax2.set_xlabel('Initial Amplitude $\theta_0$ (degrees)')
ax2.set_ylabel('Period $T$ (seconds)')
ax2.set_title('Period Lengthening with Amplitude', fontsize=10)
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('figs/fig1_5_phase_portrait_period.png')
plt.show()
